# 09 — Final Conclusions

## Objective

This chapter synthesizes the complete SECOM yield-analysis project and
answers the engineering questions defined at the beginning of the study.

The objectives are:

1. summarize the end-to-end analytical workflow;
2. answer the original research questions;
3. report the main quantitative findings;
4. state the engineering value of the proposed monitoring framework;
5. distinguish the final contribution from unsupported causal or
   production-readiness claims.

## 9.1 Engineering Problem

Semiconductor manufacturing measurements are high-dimensional, incomplete,
and often strongly imbalanced because Fail events are relatively rare.

The central project question was:

> Can anonymous process and sensor measurements be used to prioritize
> samples with elevated Fail risk while keeping engineering investigation
> workload and false alarms visible?

The project was designed as a risk-ranking and investigation-support study
rather than a conventional accuracy-maximization exercise.

## 9.2 Data Context

The analysis used the UCI SECOM dataset, containing:

- 1,567 manufacturing samples;
- 590 anonymous measurement variables;
- 1,463 Pass samples;
- 104 Fail samples;
- an overall Fail prevalence of approximately 6.64%;
- widespread missing measurements;
- timestamps but limited manufacturing metadata.

The dataset did not contain sufficient tool, chamber, lot, recipe,
product, wafer-map, or physical measurement information for confirmed
root-cause analysis.

## 9.3 End-to-End Analytical Workflow

The project followed the sequence below:

1. audit the measurement and label data;
2. characterize class imbalance, missingness, and feature quality;
3. define a leakage-safe stratified cross-validation design;
4. reproduce the UCI Kernel Ridge reference baseline;
5. compare a small set of linear and nonlinear candidate models;
6. generate out-of-fold risk scores;
7. evaluate ranking performance and threshold trade-offs;
8. identify stable anonymous candidate signals;
9. examine coefficient direction, observed distributions, and missingness;
10. translate the model outputs into investigation budgets and risk tiers;
11. build sample-level investigation context;
12. evaluate temporal variation and propose shadow-mode monitoring;
13. document limitations and future validation requirements.

## 9.4 Research Question 1

### Can the anonymous measurements provide useful Fail-risk ranking?

Yes, but the predictive ability was useful rather than sufficient for
automatic classification.

The selected Random Forest achieved:

- out-of-fold Average Precision: 0.1627;
- out-of-fold ROC-AUC: 0.7457.

The Average Precision was higher than the overall Fail prevalence of
approximately 0.0664, indicating that Fail samples were concentrated
toward the higher end of the risk ranking.

## 9.5 Research Question 2

### Which model was selected for downstream analysis?

Random Forest was selected as the primary downstream model.

Its predictive advantage over Logistic Regression was modest rather than
overwhelming. The selection was based on the combination of:

- useful out-of-fold risk ranking;
- ability to represent nonlinear relationships;
- ability to represent feature interactions;
- practical investigation-budget trade-offs;
- compatibility with validation-fold permutation importance.

The selection should not be interpreted as evidence that Random Forest is
universally superior to all other models.

## 9.6 Research Question 3

### Can the model concentrate Fail samples into a smaller investigation
queue?

Yes.

The investigation-budget analysis produced the following results:

| Investigation scope | Samples reviewed | Fail samples detected | Fail recall | Alert precision | Precision lift |
|---:|---:|---:|---:|---:|---:|
| Highest-risk 5% | 79 | 18 | 17.3% | 22.8% | 3.43 |
| Highest-risk 10% | 157 | 32 | 30.8% | 20.4% | 3.07 |
| Highest-risk 15% | 235 | 43 | 41.3% | 18.3% | 2.76 |
| Highest-risk 20% | 314 | 53 | 51.0% | 16.9% | 2.54 |
| Highest-risk 25% | 392 | 62 | 59.6% | 15.8% | 2.38 |
| Highest-risk 30% | 470 | 67 | 64.4% | 14.3% | 2.15 |

The highest-risk 5% had an observed Fail rate of 22.8%, approximately
3.43 times the overall Fail rate.

The highest-risk 20% contained 51% of all observed Fail samples.

## 9.7 Research Question 4

### Can the risk scores be translated into an engineering investigation
queue?

A retrospective three-level prototype was constructed:

- **Red:** highest-risk 5%;
- **Amber:** next 15%;
- **Routine:** remaining 80%.

The observed results were:

| Risk tier | Sample count | Fail count | Observed Fail rate | Share of all Fail cases |
|---|---:|---:|---:|---:|
| Red | 79 | 18 | 22.8% | 17.3% |
| Amber | 235 | 35 | 14.9% | 33.7% |
| Routine | 1,253 | 51 | 4.1% | 49.0% |

The monotonic decrease in Fail rate from Red to Amber to Routine showed
that the risk ranking could support engineering prioritization.

However, the Routine tier still contained 49% of all Fail samples.
Therefore, Routine does not mean guaranteed Pass. Existing production
testing and quality-control procedures must remain in place.

## 9.8 Research Question 5

### Which anonymous variables showed stable predictive evidence?

Eight variables satisfied the predefined consensus criteria:

- feature_298;
- feature_21;
- feature_59;
- feature_163;
- feature_121;
- feature_33;
- feature_79;
- feature_122.

These variables had positive mean and median permutation importance,
positive permutation importance in at least seven folds, and repeated
F-test selection.

Five signals showed agreement between their observed Pass/Fail direction
and their Logistic Regression coefficient direction:

- feature_298;
- feature_21;
- feature_59;
- feature_121;
- feature_33.

Higher observed values of these signals were generally associated with
higher predicted Fail risk.

Three variables showed directional disagreement:

- feature_163;
- feature_79;
- feature_122.

These variables were retained as context-dependent predictive signals but
were not translated into simple high-value or low-value alarm rules.

Feature_59 showed especially consistent evidence:

- F-test selection in all ten training folds;
- the largest mean F-statistic among the consensus signals;
- a fully consistent positive Logistic Regression coefficient direction;
- the largest standardized Pass/Fail median shift;
- elevated-value flag rates of 52%, 42%, and 3% in the Red, Amber, and
  Routine tiers, respectively.

This makes feature_59 a high-priority candidate for mapping to its real
measurement definition and process context.

## 9.9 Research Question 6

### Was candidate-signal missingness associated with Fail?

No reliable association was identified.

Missing-value rates for the eight consensus signals were generally low,
and none of the missingness comparisons remained statistically
significant after multiple-testing adjustment.

Missingness was therefore retained as a data-quality indicator rather
than being treated as an independent Fail-risk signal.

## 9.10 Research Question 7

### Were risk scores and alerts stable over time?

No.

Retrospective weekly monitoring showed a pronounced temporal shift.

Earlier weeks generally had:

- higher median risk scores;
- higher Red and Amber alert rates;
- higher observed Fail rates;
- more candidate-signal flags.

Later weeks generally had lower risk scores and substantially fewer
priority alerts.

This result showed that fixed global thresholds could create unstable
engineering workloads.

The time analysis was retrospective and used scores generated through
shuffled cross-validation. It did not constitute strict past-to-future
validation.

Time-based validation is therefore required before operational use.

## 9.11 Engineering Recommendation

The proposed system should be used as an investigation-prioritization
layer above existing production testing and quality-control procedures.

### Red

- highest engineering-review priority;
- inspect candidate-signal deviations;
- review available lot, tool, chamber, recipe, and temporal context;
- search for repeated or clustered related alerts.

### Amber

- secondary review when investigation capacity permits;
- escalate repeated alerts within the same manufacturing context;
- examine batch-level and time-window patterns.

### Routine

- continue existing testing and quality-control procedures;
- do not interpret as guaranteed Pass;
- retain samples for audit and false-negative review.

## 9.12 Proposed Operational Validation

Before affecting any product-disposition decision, the model should run
in shadow mode.

During shadow mode:

1. risk scores and tiers are generated and logged;
2. no product is automatically held, released, reworked, or scrapped;
3. final Pass/Fail outcomes are linked back to predictions;
4. alert volume and engineering workload are measured;
5. candidate signals are mapped to physical process context;
6. data and score drift are monitored;
7. threshold decisions are reviewed using actual investigation capacity
   and cost information.

## 9.13 Project Contributions

The main contributions of the project are:

1. a reproducible and leakage-safe cross-validation workflow for
   high-dimensional, incomplete, and imbalanced manufacturing data;
2. a model comparison based on risk ranking and Fail-detection trade-offs
   rather than accuracy alone;
3. a transparent investigation-budget analysis linking model performance
   to engineering workload;
4. a multi-method signal-identification framework combining permutation
   importance, F-test stability, coefficient direction, observed
   distributions, and missingness analysis;
5. a retrospective Red, Amber, and Routine investigation queue;
6. sample-level candidate-signal context for engineering review;
7. temporal monitoring that identified potential score and alert
   instability;
8. a shadow-mode and model-governance proposal;
9. explicit separation between predictive evidence, physical causality,
   and production readiness.

## 9.14 Final Limitations

The analysis was limited by the small number of Fail samples, high
dimensionality, missing measurements, anonymous variables, missing
manufacturing metadata, lack of external validation, and use of shuffled
rather than strict temporal cross-validation.

The model scores were not calibrated probabilities, the candidate signals
were not confirmed physical root causes, and the retrospective thresholds
were not production specifications.

These limitations prevent direct operational deployment but do not remove
the value of the analysis as an investigation-prioritization prototype.

## 9.15 Final Conclusion

This project developed a reproducible retrospective framework for
semiconductor manufacturing Fail-risk prioritization using the UCI SECOM
dataset.

A leakage-safe stratified cross-validation workflow was used to compare
reference, linear, and nonlinear models. Random Forest was selected as the
primary downstream model because it provided useful out-of-fold risk
ranking, supported nonlinear relationships and interactions, and enabled
validation-fold permutation importance.

The model achieved an out-of-fold Average Precision of 0.1627 and a
ROC-AUC of 0.7457. The highest-risk 5% of samples had an observed Fail
rate of 22.8%, approximately 3.43 times the overall Fail rate. The
highest-risk 20% contained 51% of all observed Fail samples.

Eight anonymous variables received repeated predictive and statistical
support. Five signals showed agreement between observed and multivariable
directions, while three were retained as context-dependent signals.
Feature_59 showed particularly consistent evidence across the different
analytical views.

The risk scores and candidate signals were translated into a retrospective
Red, Amber, and Routine investigation framework. Temporal monitoring
revealed substantial changes in score and alert distributions, showing
that fixed global thresholds would require further validation and
governance.

The final result is an engineering decision-support and investigation-
prioritization prototype. It is not an automatic product-disposition
system and does not identify confirmed physical root causes.

Future work should focus on restoring manufacturing context, conducting
strict temporal and external validation, running shadow-mode monitoring,
measuring real investigation costs, and validating the physical meaning
of the candidate signals.

## 9.16 Final Result Summary

| Area | Final result |
|---|---|
| Dataset | 1,567 samples, 590 anonymous measurements |
| Class distribution | 1,463 Pass, 104 Fail |
| Fail prevalence | 6.64% |
| Main evaluation | Leakage-safe stratified 10-fold OOF |
| Primary model | Random Forest |
| OOF Average Precision | 16.27% |
| OOF ROC-AUC | 74.57% |
| Highest-risk 5% | 22.8% Fail rate, 3.43× lift |
| Highest-risk 20% | 51% of all Fail samples |
| Consensus signals | 8 |
| Direction-consistent signals | 5 |
| Context-dependent signals | 3 |
| Missingness result | No reliable association for final signals |
| Monitoring finding | Strong temporal score and alert shift |
| Intended use | Investigation prioritization |
| Not supported | Automatic disposition or confirmed root cause |